In [5]:
%use coroutines
@file:DependsOn("io.github.karloti:concurrent-priority-queue:1.3.4")

In [7]:
import io.github.karloti.cpq.ConcurrentPriorityQueue

val queue = ConcurrentPriorityQueue<Int>(maxSize = 5)
queue.add(10)
queue.add(50)
queue.add(20)
queue.add(5)
queue.add(100)
queue.add(1)   // Rejected — worse than all top 5

println(queue.items.value)  // [100, 50, 20, 10, 5]

[1, 5, 10, 20, 50]


In [9]:
data class SearchResult(val id: String, val score: Int)

val queue = ConcurrentPriorityQueue<SearchResult, String>(
    maxSize = 3,
    comparator = compareByDescending { it.score },  // Higher score = higher priority
    uniqueKeySelector = { it.id }
)

queue.add(SearchResult("A", 10))
queue.add(SearchResult("B", 20))
queue.add(SearchResult("A", 30))  // Updates "A" to score 30 (better priority)
queue.add(SearchResult("A", 5))   // Rejected — existing score 30 is better
queue.add(SearchResult("C", 15))

println(queue.items.value)
// [SearchResult(id=A, score=30), SearchResult(id=B, score=20), SearchResult(id=C, score=15)]

[SearchResult(id=A, score=30), SearchResult(id=B, score=20), SearchResult(id=C, score=15)]


In [28]:
data class Task(val id: String, val deadline: Long)

val taskQueue = ConcurrentPriorityQueue<Task, String>(
    maxSize = 100,
    comparator = compareBy { it.deadline },  // Earlier deadline = higher priority
    uniqueKeySelector = { it.id }
)

taskQueue.add(Task("backup", 1735689600))
taskQueue.add(Task("critical-fix", 1735603200))

println("first = ${taskQueue.first()}") // Task(id=critical-fix, ...)
val next = taskQueue.poll() // Removes and returns the highest priority task
println("next = ${next}")


first = Task(id=critical-fix, deadline=1735603200)
next = Task(id=critical-fix, deadline=1735603200)


In [22]:
data class Task(val id: String, val priority: Long)

val queue = ConcurrentPriorityQueue<Task, String>(
    maxSize = 100,
    comparator = compareBy { it.priority },  // Earlier deadline = higher priority
    uniqueKeySelector = { it.id }
)
// Efficiently apply multiple changes without per-operation CAS overhead
val updated = queue.mutate {
    add(Task("task-1", 10))
    add(Task("task-2", 5))
    add(Task("task-3", 20))
    removeIf { it.priority > 15 }
    poll()  // Remove highest priority
}
updated.items.value
// `updated` is a new ConcurrentPriorityQueue with all changes applied

[Task(id=task-1, priority=10)]